# 🗣️ Hinglish Voice & Text Sentiment Analysis — Google Colab

Fine-tune a Transformer (DistilBERT / mBERT) on the **SentiMix** Hinglish dataset using a free Colab **T4 GPU**, then run inference.

**Before you start:** `Runtime → Change runtime type → Hardware accelerator → GPU (T4)`.

This notebook clones the project from GitHub and drives the same `src/` code used locally — nothing is duplicated.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Clone the project from GitHub

In [ ]:
REPO = "https://github.com/dexcodf/College-Project-2026.git"
BRANCH = "feature/hinglish-sentiment"

!git clone --branch $BRANCH --depth 1 $REPO
%cd College-Project-2026/hinglish-sentiment-analysis
!ls

## 3. Install dependencies
(Colab already ships torch; this installs transformers, datasets, whisper, etc.)

In [ ]:
!pip install -q -r requirements.txt
print('\n✅ Dependencies installed. Restart the runtime ONLY if Colab asks you to, then re-run from here.')

## 4. Configure the run

On a T4 GPU you can use a much larger subset and more epochs than on a laptop CPU.
Everything is controlled by environment variables — no code edits needed.

| Model | `MODEL_NAME` | Notes |
|-------|--------------|-------|
| DistilBERT (default) | `distilbert-base-multilingual-cased` | fast, ungated |
| mBERT | `bert-base-multilingual-cased` | stronger, ~2x slower |
| IndicBERT | `ai4bharat/indic-bert` | best for Hinglish, **gated** → needs `huggingface-cli login` |

In [ ]:
import os
os.environ['MODEL_NAME']  = 'bert-base-multilingual-cased'   # or distilbert-base-multilingual-cased
os.environ['SUBSET_SIZE'] = '20000'   # 20k-50k recommended on GPU
os.environ['EPOCHS']      = '4'
os.environ['BATCH_SIZE']  = '32'
os.environ['MAX_LENGTH']  = '128'

import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 5. Build dataset splits + EDA visualisations

In [ ]:
!python -m src.dataset_loader

from IPython.display import Image, display
display(Image('reports/class_distribution.png'))
display(Image('reports/text_length_distribution.png'))

## 6. Train the transformer (GPU)
Auto-evaluates on the test split when done and writes training curves.

In [ ]:
!python -m src.train

In [ ]:
from IPython.display import Image, display
display(Image('reports/training_curves.png'))
display(Image('reports/confusion_matrix_transformer.png'))

## 7. Classical baseline + comparison (TF-IDF + Logistic Regression)

In [ ]:
!python -m src.baseline

## 8. Live inference

In [ ]:
from src.inference import get_predictor
predictor = get_predictor()

samples = [
    'Bhai app bahut acha hai',
    'Yeh movie bilkul bakwaas thi, paise barbaad',
    'Theek hai, kuch khaas nahi',
]
for r in predictor.predict_batch(samples):
    print(f"{r['sentiment']:>8}  ({r['confidence']:.2f})  | {r['text']}")

## 9. (Optional) Voice input with Whisper
Upload a short WAV/MP3/M4A clip of Hinglish speech.

In [ ]:
from google.colab import files
from src.speech import transcribe
from src.inference import get_predictor

uploaded = files.upload()
for name in uploaded:
    stt = transcribe(name)
    print('Transcription:', stt['text'])
    print('Sentiment   :', get_predictor().predict(stt['text']))

## 10. (Optional) Save the trained model to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/hinglish_sentiment_model
!cp -r saved_models/best_model /content/drive/MyDrive/hinglish_sentiment_model/
print('✅ Saved to Google Drive.')